# 01a. QuartzNet + InterCTC — вспомогательный промежуточный CTC-loss

## Идея

Модель выдаёт `EncoderOutput.intermediate` — активации после блока B2 (256-канальный тензор `[B, T', 256]`).
Мы проецируем их линейным слоем `InterCTCHead` в пространство vocab и добавляем CTC-loss
на этот промежуточный выход с весом `inter_ctc_weight`.

Это заставляет энкодер уже на средних слоях формировать семантически значимые представления
и работает как регуляризация — аналогично методу Lee & Watanabe (Intermediate Loss Regularization, 2020).

**Адаптация под Wave-1 Batch:** `Batch` содержит 6 полей (`audio`, `audio_lengths`, `targets`,
`target_lengths`, `spk_ids`, `transcriptions`). `InterCTCHead` использует `out.intermediate`
из `EncoderOutput` — это поле не было удалено.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Локально: если репо уже установлен (uv sync / pip install -e .), эта ячейка — no-op.

# --- Colab ---
# !git clone --depth 1 https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git
# import sys
# sys.path.insert(0, "ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

# --- Kaggle ---
# import sys
# sys.path.insert(0, "/kaggle/working/ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

print("Deps cell — fill in your platform block above if on Colab/Kaggle.")

## Пути

`DATA_ROOT`, `CKPT_ROOT` и сопутствующие пути автоматически резолвятся в ячейке ниже относительно расположения ноутбука (`notebooks/experiments/..`). Правки требуются, только если структура репозитория изменена.

In [ ]:
# Idempotent data download: populates ../data/ with train/ dev/ test/ and
# their CSVs from the shared Google Drive ZIP. No-op if already present.
from pathlib import Path
import zipfile

import gdown

NOTEBOOK_DIR = Path.cwd()  # notebooks/experiments/
DATA_ROOT = (NOTEBOOK_DIR / ".." / "data").resolve()
ZIP_PATH = (NOTEBOOK_DIR / ".." / "data.zip").resolve()

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        gdown.download(
            url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
            output=str(ZIP_PATH),
            quiet=False,
            fuzzy=True,
        )
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT.parent)
    print(f"Extracted to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT}")


In [ ]:
# Env hardening — MUST run before `import torch` in this process.
# PYTORCH_CUDA_ALLOC_CONF reduces VRAM fragmentation on torch.compile +
# variable T batches; cudnn.benchmark=False avoids autotune thrash under
# padded, variable-length inputs; matmul_precision="high" enables TF32.
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("high")

# Limit torch.compile graph cache (dynamic=True can cache 50+ variants).
import torch._dynamo
torch._dynamo.config.cache_size_limit = 8


In [ ]:
# Paths — auto-resolved from DATA_ROOT defined in the download cell.
from pathlib import Path
import torch

TRAIN_ROOT = DATA_ROOT / "train"
DEV_ROOT = DATA_ROOT / "dev"
_TEST_DIR = DATA_ROOT / "test"
TEST_ROOT: Path | None = _TEST_DIR if _TEST_DIR.exists() else None
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = (Path.cwd() / ".." / ".." / "checkpoints" / "01a_inter_ctc").resolve()

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

MUSAN_ROOT: Path | None = Path.home() / "datasets" / "musan" / "noise"
RIR_ROOT: Path | None = Path.home() / "datasets" / "RIRS_NOISES" / "simulated_rirs"
if MUSAN_ROOT is not None and not MUSAN_ROOT.exists():
    print(f"[warn] MUSAN not found at {MUSAN_ROOT} — AddNoise disabled")
    MUSAN_ROOT = None
if RIR_ROOT is not None and not RIR_ROOT.exists():
    print(f"[warn] RIR not found at {RIR_ROOT} — RIR disabled")
    RIR_ROOT = None

In [ ]:
import gc
import json
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from gp1.data.manifest import records_from_csv
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.audio_aug_gpu import GPUAudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import save_best, load_checkpoint
from gp1.train.metrics import compute_cer, compute_cer_in_out_harmonic, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.types import AugConfig


In [ ]:
class InterCTCHead(nn.Module):
    """Auxiliary CTC head applied to intermediate encoder features.

    Adapted for Wave-1 Batch: uses out.intermediate (not a separate field).

    Args:
        d_mid: Dimension of intermediate encoder features (256 for QuartzNet10x4).
        vocab_size: Output vocabulary size (same as main head).
        blank_id: CTC blank token id (default 0).
    """

    def __init__(self, d_mid: int, vocab_size: int, blank_id: int = 0) -> None:
        super().__init__()
        self.proj = nn.Linear(d_mid, vocab_size)
        self._ctc = CTCLoss(blank_id=blank_id)

    def forward(
        self,
        mid_features: torch.Tensor,    # [B, T, d_mid]
        input_lengths: torch.Tensor,   # [B]
        targets: torch.Tensor,         # [B, U]
        target_lengths: torch.Tensor,  # [B]
    ) -> torch.Tensor:
        assert mid_features.dim() == 3, "mid_features must be [B, T, D]"
        logits = self.proj(mid_features)
        log_probs = F.log_softmax(logits.float(), dim=-1)
        return self._ctc(log_probs, targets, input_lengths, target_lengths)

In [ ]:
# --- Manifests + vocab ---
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
vocab = CharVocab()
print(f"Train: {len(train_records)} records. Dev: {len(dev_records)} records.")
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

# In-domain / out-of-domain speaker split for harmonic CER metric.
in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(
    f"dev speakers: in-domain={in_domain_count} samples, "
    f"out-of-domain={out_of_domain_count} samples"
)


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [ ]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 30,
    "grad_accum": 2,
    "subsample_factor": 2,
}

HP_GRID = {
    "d_model": [192, 256],
    "dropout": [0.05, 0.10, 0.15],
    "lr": [0.005, 0.010, 0.015],
    "weight_decay": [1e-4, 1e-3],
    "warmup_steps": [1000, 2000, 3000],
    "min_lr_ratio": [0.01, 0.05],
    "specaug_freq_mask_param": [10, 15, 20],
    "specaug_num_freq_masks": [1, 2],
    "specaug_time_mask_param": [20, 25, 35],
    "specaug_num_time_masks": [2, 5],
    "specaug_time_mask_max_ratio": [0.04, 0.05, 0.08],
    "speed_prob": [0.5, 1.0],
    "speed_factors": [(0.9, 1.0, 1.1), (0.85, 1.0, 1.15)],
    "pitch_prob": [0.0, 0.3],
    "pitch_range_semitones": [(-2.0, 2.0), (-3.0, 3.0)],
    "gain_prob": [0.3, 0.7],
    "gain_db_range": [(-4.0, 4.0), (-8.0, 8.0)],
    "vtlp_prob": [0.0, 0.3],
    "vtlp_alpha_range": [(0.9, 1.1), (0.85, 1.15)],
    "noise_prob": [0.0, 0.3],
    "noise_snr_db_range": [(10.0, 20.0), (5.0, 20.0)],
    "rir_prob": [0.0, 0.3],
    "inter_ctc_weight": [0.2, 0.3, 0.5],
}

N_TRIALS = 5
SEED = 42
random.seed(SEED)

print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


In [ ]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)
# Move tensors to shared memory so DataLoader worker processes reuse the
# same underlying buffers instead of copying on every fork.
for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()
print(f"AUDIO_CACHE: {len(AUDIO_CACHE)} entries")


In [ ]:
def _worker_init(worker_id: int) -> None:
    """1 BLAS-thread per worker + per-worker RNG seed for augmenter."""
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


In [ ]:
# DataLoader / augmenter / SpecAug — пересоздаются на каждом trial внутри Шага 5.
# Здесь только sanity-check, что записи и AUDIO_CACHE готовы.
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")
print(f"AUDIO_CACHE entries: {len(AUDIO_CACHE)}")


## Шаг 5. HP Random Search — Training Loop

Каждый trial семплирует значения из `HP_GRID`, пересоздаёт hp-зависимые объекты
(augmenter, dataset, model, optimizer, scheduler, head), запускает развёрнутый
custom training loop и сохраняет best checkpoint в `trial_dir`.


In [ ]:
mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)


In [ ]:
BATCH_SIZE = 32
DL_WORKERS = 4

trial_results = []
run_root = CKPT_ROOT / f"01a_inter_ctc_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    trial_dir = run_root / f"trial_{trial:02d}"
    trial_dir.mkdir(parents=True, exist_ok=True)

    aug_cfg = AugConfig(
        speed_factors=tuple(hp["speed_factors"]),
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        pitch_range_semitones=tuple(hp["pitch_range_semitones"]),
        gain_prob=hp["gain_prob"],
        gain_db_range=tuple(hp["gain_db_range"]),
        seed=SEED + trial,
    )
    train_augmenter = AudioAugmenter(aug_cfg)
    gpu_aug = GPUAudioAugmenter(
        samplerate=FIXED["samplerate"],
        vtlp_prob=hp["vtlp_prob"],
        vtlp_alpha_range=tuple(hp["vtlp_alpha_range"]),
        noise_prob=hp["noise_prob"],
        noise_snr_db_range=tuple(hp["noise_snr_db_range"]),
        musan_root=MUSAN_ROOT,
        rir_prob=hp["rir_prob"],
        rir_root=RIR_ROOT,
    ).to(device)
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=hp["specaug_num_freq_masks"],
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=hp["specaug_num_time_masks"],
        time_mask_max_ratio=hp["specaug_time_mask_max_ratio"],
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        target_samplerate=FIXED["samplerate"],
        augmenter=train_augmenter,
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        target_samplerate=FIXED["samplerate"],
        augmenter=None,
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=DL_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=DL_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )

    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)

    inter_head = InterCTCHead(
        d_mid=model._d_mid,
        vocab_size=vocab.vocab_size,
        blank_id=vocab.blank_id,
    ).to(device)

    ctc_loss_fn = CTCLoss(blank_id=vocab.blank_id)
    all_params = list(model.parameters()) + list(inter_head.parameters())
    optimizer = build_novograd(
        all_params,
        lr=hp["lr"],
        betas=(0.95, 0.5),
        weight_decay=hp["weight_decay"],
    )
    steps_per_epoch = math.ceil(len(train_loader) / FIXED["grad_accum"])
    total_steps = steps_per_epoch * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
        min_lr_ratio=hp["min_lr_ratio"],
    )

    best_cer = float("inf")
    best_ckpt_path = None

    for epoch in tqdm(range(FIXED["max_epochs"]), desc=f"trial {trial} epochs"):
        model.train()
        inter_head.train()
        spec_aug.train()
        grad_step = 0
        optimizer.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", leave=False)):
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            targets = batch.targets.to(device)
            target_lengths = batch.target_lengths.to(device)

            if model.training:
                audio = gpu_aug(audio, audio_lengths)

            with torch.no_grad():
                mel = mel_extractor(audio)

            mel_lengths = (
                (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
            ).clamp(max=mel.size(-1))

            mel = spec_aug(mel, mel_lengths)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                out = model(mel, mel_lengths)

            with torch.autocast(device_type=device.type, enabled=False):
                loss_main = ctc_loss_fn(
                    out.log_probs.float(), targets, out.output_lengths, target_lengths
                )
                if out.intermediate is not None:
                    loss_inter = inter_head(
                        out.intermediate, out.output_lengths, targets, target_lengths
                    )
                else:
                    loss_inter = torch.tensor(0.0, device=device)

            loss = loss_main + hp["inter_ctc_weight"] * loss_inter

            (loss / FIXED["grad_accum"]).backward()
            grad_step += 1
            if grad_step % FIXED["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        model.eval()
        inter_head.eval()
        spec_aug.eval()
        all_refs, all_hyps, all_spks = [], [], []

        with torch.no_grad():
            for batch in dev_loader:
                audio = batch.audio.to(device)
                audio_lengths = batch.audio_lengths.to(device)
                mel = mel_extractor(audio)
                mel_lengths = (
                    (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
                ).clamp(max=mel.size(-1))
                out = model(mel, mel_lengths)
                hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
                all_refs.extend(batch.transcriptions)
                all_hyps.extend(hyps)
                all_spks.extend(batch.spk_ids)

        ref_words = [digits_to_words(r) for r in all_refs]
        val_cer = compute_cer(ref_words, all_hyps)
        in_cer, out_cer, hm_cer = compute_cer_in_out_harmonic(
            ref_words, all_hyps, all_spks, in_domain_speakers
        )

        tqdm.write(
            f"trial {trial} epoch {epoch:3d} | hm_cer={hm_cer:.4f} "
            f"(in={in_cer:.4f} out={out_cer:.4f}) | val_cer={val_cer:.4f}"
        )

        if hm_cer < best_cer:
            best_cer = hm_cer
            best_ckpt_path = save_best(
                model,
                meta=dict(
                    arch="quartznet10x4_inter_ctc",
                    hparams={**FIXED, **hp},
                    hm_cer=hm_cer,
                    val_cer=val_cer,
                    in_cer=in_cer,
                    out_cer=out_cer,
                    epoch=epoch,
                    trial=trial,
                ),
                ckpt_dir=trial_dir,
            )
            tqdm.write(f"  Checkpoint saved: {best_ckpt_path}")

    trial_results.append({
        "trial": trial,
        "hp": hp,
        "best_monitored": best_cer,
        "best_ckpt_path": best_ckpt_path,
    })

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"trial {trial}: peak VRAM = {peak_gb:.2f} GB")

    del model, inter_head, optimizer, scheduler, ctc_loss_fn
    del train_loader, dev_loader, train_ds, dev_ds
    del train_augmenter, gpu_aug, spec_aug
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

print("\nHP search complete.")


## Шаг 6. Сводный отчёт трайлов


In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_monitored"])
print(f"{'trial':>5} {'best_cer':>10} {'lr':>8} {'dropout':>8} {'d_model':>8} {'inter_w':>8}")
print("-" * 60)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_monitored']:>10.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
        f" {hp['inter_ctc_weight']:>8.2f}"
    )


## Шаг 7. Валидация лучшей модели на dev (greedy)

Лучший trial по `harmonic_in_out_cer` загружается из ckpt и оценивается на dev
без InterCTC-головы (single forward через `model`).


In [ ]:
best_result = trial_results_sorted[0]
best_hp = best_result["hp"]
best_ckpt = best_result["best_ckpt_path"]
print(f"Best trial: #{best_result['trial']}, hm_cer={best_result['best_monitored']:.4f}")
print(f"ckpt: {best_ckpt}")

model = QuartzNet10x4(
    vocab_size=vocab.vocab_size,
    d_model=best_hp["d_model"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    augmenter=None,
    target_samplerate=FIXED["samplerate"],
)
dev_loader_eval = DataLoader(
    dev_ds_eval,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=4,
)

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(digits_to_words(t) for t in batch.transcriptions)
        spks.extend(batch.spk_ids)

cer = compute_cer(refs, predictions)
per_spk = compute_per_speaker_cer(refs, predictions, spks)
print(f"Dev CER (greedy): {cer:.4f}")
print(f"Per-speaker CER: {per_spk}")
